# Modulo 4 · Lezione 5 — Python + SQL

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Argomento | Strumento |
|---|------|-------------|
| 1 | Connessione e cursore | `sqlite3` (stdlib) |
| 2 | Eseguire istruzioni | `execute`, `executemany`, `executescript` |
| 3 | Recuperare i risultati | `fetchone`, `fetchall`, `fetchmany` |
| 4 | Parametri sicuri | `?` — prevenire SQL injection |
| 5 | Context manager | `with conn:` per le transazioni |
| 6 | Integrazione con pandas | `read_sql_query`, `to_sql` |
| 7 | SQLAlchemy — Core | Engine, connessione, testo SQL |
| 8 | SQLAlchemy — ORM | Modelli, sessione, CRUD |
| 9 | Pratica | Pipeline di analisi completa |

---
## 1. `sqlite3` — il modulo della libreria standard

Non devi installare nulla. `sqlite3` è incluso in Python.

```python
import sqlite3

# File su disco
conn = sqlite3.connect("mi_base.db")

# Database in memoria (si perde alla chiusura)
conn = sqlite3.connect(":memory:")

# Creare il cursore
cur = conn.cursor()

# Eseguire
cur.execute("SELECT ...")

# Confermare le modifiche
conn.commit()

# Chiudere
conn.close()
```

In [ ]:
import sqlite3
from pathlib import Path

DB = Path.home() / "demo_python_sql.db"
conn = sqlite3.connect(DB)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

# Creare la tabella e popolarla
cur.executescript("""
    DROP TABLE IF EXISTS tareas;
    CREATE TABLE tareas (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo      TEXT    NOT NULL,
        completada  INTEGER DEFAULT 0,
        prioridad   TEXT    DEFAULT 'normal'
    );
""")

cur.executemany(
    "INSERT INTO tareas (titulo, prioridad) VALUES (?, ?)",
    [
        ("Revisar correo",         "normal"),
        ("Preparar informe",       "alta"),
        ("Llamar al cliente",      "alta"),
        ("Actualizar documentación","baja"),
        ("Sprint planning",        "alta"),
        ("Code review",            "normal"),
    ]
)
conn.commit()
print(f"Database creato in {DB}")

---
## 2 e 3. Eseguire e recuperare i risultati

```python
cur.execute("SELECT ...")

cur.fetchone()          # → tuple con la prima riga, o None
cur.fetchall()          # → list[tuple] con tutte le righe
cur.fetchmany(n)        # → list[tuple] con le successive n righe

# Iterare direttamente sul cursore
for fila in cur.execute("SELECT ..."):
    print(fila)

# Accesso per nome di colonna (invece che per indice)
conn.row_factory = sqlite3.Row
cur.execute("SELECT nombre, precio FROM productos")
fila = cur.fetchone()
print(fila['nombre'], fila['precio'])
```

In [ ]:
# fetchone
cur.execute("SELECT * FROM tareas WHERE prioridad = 'alta'")
primera = cur.fetchone()
print("fetchone:", primera)

# fetchall
cur.execute("SELECT id, titulo FROM tareas ORDER BY id")
todas = cur.fetchall()
print("\nfetchall:", todas)

# Iterare sul cursore
print("\nIterazione sul cursore:")
for id_, titulo, completada, prioridad in cur.execute("SELECT * FROM tareas ORDER BY prioridad"):
    estado = "✅" if completada else "⬜"
    print(f"  {estado} [{prioridad:6s}] {titulo}")

# Accesso per nome con row_factory
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute("SELECT * FROM tareas LIMIT 2")
print("\nrow_factory — accesso per nome:")
for fila in cur.fetchall():
    print(f"  id={fila['id']} titulo='{fila['titulo']}' prioridad={fila['prioridad']}")

---
## 4. Parametri sicuri — prevenire SQL injection

**Non costruire mai** istruzioni SQL con f-string o concatenazione di stringhe a partire da dati inseriti dall'utente.

```python
# ❌ PERICOLOSO — SQL injection
titulo = input("Título:")  # l'utente scrive: ' OR 1=1 --
cur.execute(f"SELECT * FROM tareas WHERE titulo = '{titulo}'")
#           → SELECT * FROM tareas WHERE titulo = '' OR 1=1 --'
#             Restituisce TUTTE le righe

# ✅ SICURO — parametro ?
cur.execute("SELECT * FROM tareas WHERE titulo = ?", (titulo,))
# sqlite3 esegue l'escape del valore automaticamente
```

**Regola d'oro:** i dati vanno sempre passati come parametri, mai interpolati nell'SQL.

In [ ]:
# Cercare per priorità (parametro sicuro)
def buscar_por_prioridad(conn, prioridad):
    cur = conn.cursor()
    cur.execute("SELECT id, titulo FROM tareas WHERE prioridad = ?", (prioridad,))
    return cur.fetchall()

for p in ['alta', 'normal', 'baja']:
    resultado = buscar_por_prioridad(conn, p)
    print(f"\nPriorità '{p}':")
    for fila in resultado:
        print(f"  [{fila[0]}] {fila[1]}")

# Inserire con parametri
nueva = ("Desplegar a producción", "alta")
cur.execute("INSERT INTO tareas (titulo, prioridad) VALUES (?, ?)", nueva)
nuevo_id = cur.lastrowid
conn.commit()
print(f"\nNuova attività inserita con id={nuevo_id}")

---
## 5. Context manager per le transazioni

```python
# with conn funziona come blocco di transazione
with conn:
    conn.execute("UPDATE ...")
    conn.execute("INSERT ...")
# Se non ci sono eccezioni → commit automatico
# Se c'è un'eccezione      → rollback automatico
```

Puoi anche aprire e chiudere la connessione con `with`:

```python
with sqlite3.connect("mi_bd.db") as conn:
    conn.execute("INSERT ...")
# conn si chiude automaticamente all'uscita dal blocco
```

In [ ]:
# Transazione riuscita
with conn:
    conn.execute("UPDATE tareas SET completada = 1 WHERE id = 1")
    conn.execute("UPDATE tareas SET completada = 1 WHERE id = 2")
print("Transazione completata")

# Transazione fallita → rollback
import contextlib

try:
    with conn:
        conn.execute("UPDATE tareas SET completada = 1 WHERE id = 3")
        raise ValueError("Qualcosa è andato storto")  # simuliamo un errore
        conn.execute("UPDATE tareas SET completada = 1 WHERE id = 4")
except ValueError as e:
    print(f"Errore catturato: {e} → rollback automatico")

# Verifichiamo lo stato
cur.execute("SELECT id, titulo, completada FROM tareas")
for fila in cur.fetchall():
    print(f"  id={fila[0]} completada={fila[2]} {fila[1]}")

---
## 6. Integrazione con pandas

```python
import pandas as pd

# Leggere il risultato SQL come DataFrame
df = pd.read_sql_query("SELECT * FROM tabla WHERE ...", conn)
df = pd.read_sql_query("SELECT * FROM tabla WHERE id = ?", conn, params=(5,))

# Scrivere un DataFrame su una tabella SQL
df.to_sql(
    name='nombre_tabla',
    con=conn,
    if_exists='replace',  # 'replace' | 'append' | 'fail'
    index=False
)
```

In [ ]:
import pandas as pd

# Leggere da SQLite a DataFrame
df_tareas = pd.read_sql_query(
    "SELECT * FROM tareas ORDER BY prioridad, id",
    conn
)
print("DataFrame da SQLite:")
display(df_tareas)

# Analisi con pandas
print("\nRiepilogo per priorità:")
display(df_tareas.groupby('prioridad').agg(
    total=('id', 'count'),
    completadas=('completada', 'sum')
).reset_index())

# Scrivere un DataFrame su SQL
nuevas = pd.DataFrame([
    {'titulo': 'Revisar pull requests', 'prioridad': 'alta',   'completada': 0},
    {'titulo': 'Actualizar dependencias','prioridad': 'baja',   'completada': 0},
    {'titulo': 'Escribir tests',          'prioridad': 'normal','completada': 0},
])

nuevas.to_sql('tareas', conn, if_exists='append', index=False)
print(f"\nTotale attività ora: {pd.read_sql_query('SELECT COUNT(*) AS n FROM tareas', conn).iloc[0]['n']}")

---
## 7. SQLAlchemy — Core

SQLAlchemy è la libreria per database più usata in Python. Ha due livelli:

```
SQLAlchemy ORM   ← livello alto: oggetti Python ↔ tabella SQL
SQLAlchemy Core  ← livello medio: SQL espressivo in Python
      ↓
DB API 2.0       ← livello basso: sqlite3, psycopg2, ecc.
```

**Installazione** (se non ce l'hai):
```bash
pip install sqlalchemy
```

In [ ]:
try:
    import sqlalchemy
    print(f"SQLAlchemy {sqlalchemy.__version__} disponibile")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "sqlalchemy"], check=True)
    import sqlalchemy
    print(f"SQLAlchemy {sqlalchemy.__version__} installato")

In [ ]:
from sqlalchemy import create_engine, text
from pathlib import Path

# L'engine è la connessione al motore del DB
engine = create_engine(f"sqlite:///{Path.home() / 'tienda.db'}")

# Eseguire SQL testuale con SQLAlchemy Core
with engine.connect() as con:
    # SELECT
    resultado = con.execute(text("SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 5"))
    print("Top 5 prodotti:")
    for fila in resultado:
        print(f"  {fila.nombre:30s} {fila.precio:>8.2f}€")

    # SELECT con parametri (usando :nombre)
    resultado = con.execute(
        text("SELECT nombre, stock FROM productos WHERE categoria_id = :cat"),
        {"cat": 2}
    )
    print("\nProdotti di Informatica:")
    for fila in resultado:
        print(f"  {fila.nombre:30s} stock={fila.stock}")

---
## 8. SQLAlchemy — ORM

L'ORM (Object-Relational Mapper) mappa le tabelle SQL su classi Python. Ogni riga diventa un oggetto.

```python
# Tabella SQL            ↔     Classe Python
# Riga                  ↔     Istanza
# Colonna               ↔     Attributo
# INSERT                ↔     session.add(oggetto)
# SELECT                ↔     session.query(Classe).filter(...)
# UPDATE                ↔     oggetto.attributo = nuovo_valore
# DELETE                ↔     session.delete(oggetto)
# COMMIT                ↔     session.commit()
```

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session

# Motore (database in memoria per questa demo)
engine_orm = create_engine("sqlite:///:memory:", echo=False)
Base = declarative_base()

# Definire i modelli (classi = tabelle)
class Categoria(Base):
    __tablename__ = "categorias"
    id       = Column(Integer, primary_key=True)
    nombre   = Column(String, nullable=False)
    productos = relationship("Producto", back_populates="categoria")

    def __repr__(self):
        return f"<Categoria id={self.id} nombre='{self.nombre}'>"

class Producto(Base):
    __tablename__ = "productos"
    id           = Column(Integer, primary_key=True)
    nombre       = Column(String, nullable=False)
    precio       = Column(Float, nullable=False)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey("categorias.id"))
    categoria    = relationship("Categoria", back_populates="productos")

    def __repr__(self):
        return f"<Producto '{self.nombre}' {self.precio}€>"

# Creare le tabelle
Base.metadata.create_all(engine_orm)

# CRUD con sessione
with Session(engine_orm) as session:
    # CREATE
    electronica = Categoria(nombre="Electrónica")
    informatica = Categoria(nombre="Informática")
    session.add_all([electronica, informatica])
    session.flush()  # assegna gli ID senza ancora fare il commit

    productos = [
        Producto(nombre="Smartphone",  precio=599.99, stock=50, categoria=electronica),
        Producto(nombre="Auriculares", precio=89.99,  stock=120, categoria=electronica),
        Producto(nombre="Portátil",    precio=1199.0, stock=30,  categoria=informatica),
        Producto(nombre="Teclado",     precio=85.0,   stock=75,  categoria=informatica),
    ]
    session.add_all(productos)
    session.commit()

    # READ — query di base
    todos = session.query(Producto).all()
    print("Tutti i prodotti:")
    for p in todos:
        print(f"  {p.nombre:15s} {p.precio:>8.2f}€  cat={p.categoria.nombre}")

    # READ — filtro
    caros = session.query(Producto).filter(Producto.precio > 100).order_by(Producto.precio.desc()).all()
    print("\nProdotti > 100€:", caros)

    # UPDATE
    smartphone = session.query(Producto).filter_by(nombre="Smartphone").first()
    smartphone.precio = 549.99
    smartphone.stock -= 5
    session.commit()
    print(f"\nSmartphone aggiornato: {smartphone.precio}€, stock={smartphone.stock}")

    # DELETE
    auriculares = session.query(Producto).filter_by(nombre="Auriculares").first()
    session.delete(auriculares)
    session.commit()
    print(f"\nTotale prodotti dopo DELETE: {session.query(Producto).count()}")

---
## Quando usare ciascuno strumento

```
sqlite3 (stdlib)
  ✅ Script semplici, strumenti interni
  ✅ Nessuna dipendenza esterna
  ✅ Solo SQLite
  ⚠️ SQL manuale, più verboso

SQLAlchemy Core
  ✅ SQL espressivo in Python
  ✅ Portabilità tra motori (SQLite, PostgreSQL, MySQL...)
  ✅ Più controllo rispetto all'ORM

SQLAlchemy ORM
  ✅ Applicazioni complesse con molte entità correlate
  ✅ Codice più manutenibile e testabile
  ✅ Migrazioni con Alembic
  ⚠️ Curva di apprendimento iniziale
  ⚠️ Può nascondere query inefficienti

pandas + read_sql_query
  ✅ Analisi dati ed ETL
  ✅ Integrazione con numpy, matplotlib, scikit-learn
```

---
## 9. Pratica — Pipeline di analisi

---

### Esercizio 1 — CRUD completo con sqlite3

**Consegna:** Implementa una classe `GestorTareas` che usi `sqlite3` per:

- `crear(titulo, prioridad='normal')` → inserisce e restituisce l'id.
- `listar(prioridad=None)` → elenca tutte (o filtrate per priorità).
- `completar(id)` → segna come completata.
- `eliminar(id)` → elimina l'attività.
- `resumen()` → dict con totale, completate, in sospeso per priorità.

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import sqlite3
from pathlib import Path

class GestorTareas:
    def __init__(self, db_path=":memory:"):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS tareas (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                titulo     TEXT NOT NULL,
                completada INTEGER DEFAULT 0,
                prioridad  TEXT DEFAULT 'normal'
            )
        """)
        self.conn.commit()

    def crear(self, titulo, prioridad='normal'):
        cur = self.conn.execute(
            "INSERT INTO tareas (titulo, prioridad) VALUES (?, ?)",
            (titulo, prioridad)
        )
        self.conn.commit()
        return cur.lastrowid

    def listar(self, prioridad=None):
        if prioridad:
            cur = self.conn.execute(
                "SELECT * FROM tareas WHERE prioridad = ? ORDER BY id", (prioridad,)
            )
        else:
            cur = self.conn.execute("SELECT * FROM tareas ORDER BY prioridad DESC, id")
        return [dict(r) for r in cur.fetchall()]

    def completar(self, id_):
        self.conn.execute("UPDATE tareas SET completada = 1 WHERE id = ?", (id_,))
        self.conn.commit()

    def eliminar(self, id_):
        self.conn.execute("DELETE FROM tareas WHERE id = ?", (id_,))
        self.conn.commit()

    def resumen(self):
        cur = self.conn.execute("""
            SELECT
                COUNT(*)                AS total,
                SUM(completada)         AS completadas,
                SUM(1 - completada)     AS pendientes,
                SUM(prioridad='alta')   AS alta,
                SUM(prioridad='normal') AS normal,
                SUM(prioridad='baja')   AS baja
            FROM tareas
        """)
        return dict(cur.fetchone())


# Test
g = GestorTareas()
g.crear("Diseñar API", "alta")
g.crear("Escribir tests", "alta")
g.crear("Revisar docs", "baja")
g.crear("Daily standup", "normal")
g.completar(1)
g.eliminar(3)
print("Attività:", g.listar())
print("Riepilogo:", g.resumen())
```
</details>

---
### Esercizio 2 — Pipeline ETL con pandas

**Consegna:** Implementa la funzione `etl_ventas(conn_tienda)` che:

1. Legga da `tienda.db` tutte le righe d'ordine con i dati collegati (cliente, prodotto, categoria).
2. Calcoli una colonna `subtotal = cantidad × precio_ud`.
3. Generi un riepilogo per categoria: totale venduto, unità, importo.
4. Salvi il riepilogo in una nuova tabella `resumen_ventas` nello stesso DB.

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3, pandas as pd
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

def etl_ventas(conn):
    # 1. Estrarre
    df = pd.read_sql_query("""
        SELECT c.nombre  AS cliente,
               cat.nombre AS categoria,
               prod.nombre AS producto,
               pp.cantidad, pp.precio_ud
        FROM pedido_productos pp
        JOIN pedidos   p    ON pp.pedido_id   = p.id
        JOIN clientes  c    ON p.cliente_id   = c.id
        JOIN productos prod ON pp.producto_id = prod.id
        JOIN categorias cat ON prod.categoria_id = cat.id
    """, conn)

    # 2. Trasformare
    df['subtotal'] = df['cantidad'] * df['precio_ud']

    # 3. Aggregare
    resumen = df.groupby('categoria').agg(
        total_lineas=('subtotal', 'count'),
        unidades=('cantidad', 'sum'),
        importe=('subtotal', 'sum')
    ).round(2).reset_index()

    print("Riepilogo vendite per categoria:")
    display(resumen)

    # 4. Caricare
    resumen.to_sql('resumen_ventas', conn, if_exists='replace', index=False)
    print("\nTabella 'resumen_ventas' salvata nel DB")
    return resumen

conn = sqlite3.connect(Path.home() / "tienda.db")
resumen = etl_ventas(conn)
conn.close()
```
</details>

---
### Esercizio 3 — SQL injection

**Consegna:** Dimostra il pericolo dell'interpolazione di stringhe e come i parametri lo prevengono:

1. Crea una funzione `login_inseguro(conn, usuario, password)` con SQL interpolato.
2. Mostra come l'input `' OR 1=1 --` fa sì che vengano restituiti tutti gli utenti.
3. Implementa `login_seguro(conn, usuario, password)` con `?` e dimostra che l'iniezione fallisce.
4. Spiega nei commenti perché funziona la difesa.

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import sqlite3

# DB di demo
conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE usuarios (id INTEGER PRIMARY KEY, nombre TEXT, password TEXT);
    INSERT INTO usuarios VALUES (1, 'admin', 'secreto123');
    INSERT INTO usuarios VALUES (2, 'ana',   'pass456');
""")

# ❌ VERSIONE NON SICURA
def login_inseguro(conn, usuario, password):
    # L'SQL viene costruito concatenando stringhe: pericoloso
    sql = f"SELECT * FROM usuarios WHERE nombre='{usuario}' AND password='{password}'"
    print(f"SQL eseguito: {sql}")
    return conn.execute(sql).fetchall()

print("=== Attacco SQL injection ===")
# Input malevolo: chiude la stringa e aggiunge OR 1=1 per aggirare il filtro
resultado = login_inseguro(conn, "' OR 1=1 --", "cualquier_cosa")
print(f"Utenti restituiti: {len(resultado)} → {resultado}")
# L'attaccante ottiene TUTTI gli utenti senza conoscere nessuna password

# ✅ VERSIONE SICURA
def login_seguro(conn, usuario, password):
    # I ? sono segnaposto; sqlite3 esegue l'escape dei valori automaticamente
    # Non possono mai essere usati per iniettare SQL perché NON vengono interpolati come codice
    sql = "SELECT * FROM usuarios WHERE nombre=? AND password=?"
    return conn.execute(sql, (usuario, password)).fetchall()

print("\n=== Versione sicura ===")
resultado = login_seguro(conn, "' OR 1=1 --", "cualquier_cosa")
print(f"Utenti restituiti: {len(resultado)} → {resultado}")
# Restituisce [] perché cerca letteralmente un utente chiamato "' OR 1=1 --"

resultado_ok = login_seguro(conn, "admin", "secreto123")
print(f"Login corretto: {resultado_ok}")
```
</details>

---
### Esercizio 4 — ORM di base

**Consegna:** Con SQLAlchemy ORM, definisci i modelli `Libro` e `Autor` (relazione 1:N). Inserisci cinque autori e dieci libri. Poi interroga:
1. Tutti i libri di un dato autore.
2. Autori che hanno più di due libri.

In [ ]:
# ✏️ IL TUO CODICE QUI
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, func
from sqlalchemy.orm import declarative_base, relationship, Session

engine = create_engine("sqlite:///:memory:")
Base   = declarative_base()

class Autor(Base):
    __tablename__ = "autores"
    id     = Column(Integer, primary_key=True)
    nombre = Column(String, nullable=False)
    libros = relationship("Libro", back_populates="autor")

class Libro(Base):
    __tablename__ = "libros"
    id        = Column(Integer, primary_key=True)
    titulo    = Column(String, nullable=False)
    precio    = Column(Float)
    autor_id  = Column(Integer, ForeignKey("autores.id"))
    autor     = relationship("Autor", back_populates="libros")

Base.metadata.create_all(engine)

with Session(engine) as s:
    autores = [
        Autor(nombre="García Márquez"),
        Autor(nombre="Isabel Allende"),
        Autor(nombre="Borges"),
        Autor(nombre="Cortázar"),
        Autor(nombre="Vargas Llosa"),
    ]
    s.add_all(autores)
    s.flush()

    libros = [
        Libro(titulo="Cien años de soledad",    precio=18.5, autor=autores[0]),
        Libro(titulo="El amor en los tiempos",  precio=16.0, autor=autores[0]),
        Libro(titulo="El coronel no tiene",      precio=12.0, autor=autores[0]),
        Libro(titulo="La casa de los espíritus",precio=17.0, autor=autores[1]),
        Libro(titulo="Eva Luna",                precio=14.0, autor=autores[1]),
        Libro(titulo="Ficciones",               precio=15.0, autor=autores[2]),
        Libro(titulo="El Aleph",                precio=13.0, autor=autores[2]),
        Libro(titulo="Rayuela",                 precio=20.0, autor=autores[3]),
        Libro(titulo="Historias de Cronopios",  precio=11.0, autor=autores[3]),
        Libro(titulo="La ciudad y los perros",  precio=16.0, autor=autores[4]),
    ]
    s.add_all(libros)
    s.commit()

    # 1. Libri di García Márquez
    gm = s.query(Autor).filter_by(nombre="García Márquez").first()
    print(f"Libri di {gm.nombre}:")
    for l in gm.libros:
        print(f"  {l.titulo} — {l.precio}€")

    # 2. Autori con più di 2 libri
    print("\nAutori con > 2 libri:")
    for autor, total in (
        s.query(Autor, func.count(Libro.id).label('total'))
         .join(Libro)
         .group_by(Autor.id)
         .having(func.count(Libro.id) > 2)
         .all()
    ):
        print(f"  {autor.nombre}: {total} libri")
```
</details>

---
### Esercizio 5 (sfida) — Repository generico

**Consegna:** Implementa una classe `Repositorio` generica con SQLAlchemy ORM che funzioni con qualsiasi modello:

```python
repo = Repositorio(session, Producto)
repo.crear(nombre="Teclado", precio=85.0, stock=50, categoria_id=1)
repo.obtener(1)           # per id
repo.listar()             # tutti
repo.filtrar(precio=85.0) # per attributo esatto
repo.actualizar(1, precio=79.0)
repo.eliminar(1)
repo.contar()
```

Provalo con i modelli `Categoria` e `Producto` della demo precedente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session

engine = create_engine("sqlite:///:memory:")
Base   = declarative_base()

class Categoria(Base):
    __tablename__ = "categorias"
    id      = Column(Integer, primary_key=True)
    nombre  = Column(String, nullable=False)
    productos = relationship("Producto", back_populates="categoria")
    def __repr__(self): return f"<Categoria {self.nombre}>"

class Producto(Base):
    __tablename__ = "productos"
    id           = Column(Integer, primary_key=True)
    nombre       = Column(String, nullable=False)
    precio       = Column(Float)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey("categorias.id"))
    categoria    = relationship("Categoria", back_populates="productos")
    def __repr__(self): return f"<Producto '{self.nombre}' {self.precio}€>"

Base.metadata.create_all(engine)

class Repositorio:
    def __init__(self, session, modelo):
        self.session = session
        self.modelo  = modelo

    def crear(self, **kwargs):
        obj = self.modelo(**kwargs)
        self.session.add(obj)
        self.session.commit()
        return obj

    def obtener(self, id_):
        return self.session.get(self.modelo, id_)

    def listar(self):
        return self.session.query(self.modelo).all()

    def filtrar(self, **kwargs):
        return self.session.query(self.modelo).filter_by(**kwargs).all()

    def actualizar(self, id_, **kwargs):
        obj = self.obtener(id_)
        if obj:
            for k, v in kwargs.items():
                setattr(obj, k, v)
            self.session.commit()
        return obj

    def eliminar(self, id_):
        obj = self.obtener(id_)
        if obj:
            self.session.delete(obj)
            self.session.commit()

    def contar(self):
        return self.session.query(self.modelo).count()


with Session(engine) as s:
    repo_cat  = Repositorio(s, Categoria)
    repo_prod = Repositorio(s, Producto)

    cat = repo_cat.crear(nombre="Electrónica")
    repo_prod.crear(nombre="Smartphone",  precio=599.0, stock=50, categoria_id=cat.id)
    repo_prod.crear(nombre="Auriculares", precio=89.0,  stock=100, categoria_id=cat.id)
    repo_prod.crear(nombre="Cargador",    precio=29.0,  stock=200, categoria_id=cat.id)

    print("Tutti:", repo_prod.listar())
    print("Filtrato stock=100:", repo_prod.filtrar(stock=100))
    repo_prod.actualizar(1, precio=549.0)
    print("Dopo l'aggiornamento:", repo_prod.obtener(1))
    repo_prod.eliminar(3)
    print("Totale dopo l'eliminazione:", repo_prod.contar())
```
</details>

---
## Riepilogo della lezione

```python
# ── sqlite3 ────────────────────────────────────────────────────
import sqlite3
conn = sqlite3.connect("bd.db")
conn.row_factory = sqlite3.Row
cur  = conn.cursor()
cur.execute("SELECT * FROM t WHERE id = ?", (5,))   # ← parametri SEMPRE
cur.fetchone() · cur.fetchall() · cur.fetchmany(n)
with conn:   # transazione automatica
    conn.execute("INSERT ...")

# ── pandas ─────────────────────────────────────────────────────
df = pd.read_sql_query(sql, conn, params=(val,))
df.to_sql('tabla', conn, if_exists='replace', index=False)

# ── SQLAlchemy Core ────────────────────────────────────────────
engine = create_engine("sqlite:///bd.db")
with engine.connect() as con:
    con.execute(text("SELECT ..."), {"param": valor})

# ── SQLAlchemy ORM ─────────────────────────────────────────────
class Modelo(Base):
    __tablename__ = "tabla"
    id     = Column(Integer, primary_key=True)
    nombre = Column(String)

with Session(engine) as s:
    s.add(Modelo(nombre="X"))
    s.commit()
    obj = s.get(Modelo, 1)
    objs = s.query(Modelo).filter(Modelo.nombre == "X").all()
```

## Cosa vedremo nella prossima lezione?

Nella **Lezione 6** usciamo dal mondo relazionale: database NoSQL con MongoDB e `pymongo`.

In [ ]:
import os
from pathlib import Path
db = Path.home() / "demo_python_sql.db"
if db.exists():
    os.remove(db)
print("Pulizia completata")